# Running Experiments from EXPERIMENTS.md

This notebook provides an automated way to run the experiments described in EXPERIMENTS.md,
analyze results, and generate publication-quality visualizations.

## Experiments Covered

1. **Baseline Comparison** - Compare agents with/without internal dimensions
2. **Dimension Scaling** - Test different internal dimension sizes
3. **Intrinsic Motivation** - Evaluate curiosity-driven exploration
4. **Multi-Agent Social** - Test cooperation and mutual awareness
5. **Consciousness Metrics** - Measure R_ω, R_ψ, and φ

## Features

- ✅ Automated experiment execution
- ✅ Real-time progress tracking
- ✅ Statistical analysis and comparisons
- ✅ Publication-ready plots and tables
- ✅ Hypothesis testing

In [ ]:
# Setup
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add src to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import yaml
import json
from tqdm.notebook import tqdm
from IPython.display import display, clear_output, Markdown
from scipy import stats

from agents.ppo import PPOAgent
from environments.gridworld import GridWorld, TwoRoomGridWorld
from environments.sparse_reward import SparseRewardWrapper
from environments.social import PrisonersDilemma, IteratedPrisonersDilemma
from core.consciousness import ConsciousnessMetrics

# Set style
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300

print("✓ All imports successful!")

## Experiment Runner

This class handles running experiments with different configurations:

In [ ]:
class ExperimentRunner:
    """Run experiments and collect results."""
    
    def __init__(self, output_dir="../data/notebook_experiments"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.results = []
    
    def run_experiment(
        self,
        name,
        env,
        agent_config,
        num_episodes=100,
        max_steps=200,
        eval_episodes=10,
    ):
        """Run a single experiment."""
        print(f"\n{'='*60}")
        print(f"Running: {name}")
        print(f"{'='*60}")
        
        # Create agent
        agent = PPOAgent(
            observation_space=env.observation_space,
            action_space=env.action_space,
            **agent_config
        )
        
        # Training metrics
        metrics = {
            'episode': [],
            'reward': [],
            'episode_length': [],
            'R_omega': [],
            'R_psi': [],
            'phi': [],
        }
        
        consciousness = ConsciousnessMetrics(agent_config.get('internal_dim', 12))
        
        # Training loop
        for episode in tqdm(range(num_episodes), desc="Training"):
            obs, _ = env.reset()
            episode_reward = 0
            episode_x12 = []
            episode_m12 = []
            actions = []
            
            for step in range(max_steps):
                with torch.no_grad():
                    obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
                    action_logits, value, x12, m12 = agent.policy(obs_tensor)
                    action_dist = torch.distributions.Categorical(logits=action_logits)
                    action = action_dist.sample()
                
                next_obs, reward, terminated, truncated, _ = env.step(action.item())
                done = terminated or truncated
                
                episode_x12.append(x12.squeeze().numpy())
                episode_m12.append(m12.squeeze().numpy())
                actions.append(action.item())
                episode_reward += reward
                obs = next_obs
                
                if done:
                    break
            
            # Compute consciousness metrics
            x12_array = np.array(episode_x12)
            m12_array = np.array(episode_m12)
            actions_array = np.array(actions)
            
            R_omega = consciousness.compute_R_omega(x12_array)
            R_psi = consciousness.compute_R_psi(m12_array, actions_array)
            phi = consciousness.compute_phi(x12_array)
            
            # Store metrics
            metrics['episode'].append(episode)
            metrics['reward'].append(episode_reward)
            metrics['episode_length'].append(step + 1)
            metrics['R_omega'].append(R_omega)
            metrics['R_psi'].append(R_psi)
            metrics['phi'].append(phi)
        
        # Store results
        result = {
            'name': name,
            'config': agent_config,
            'metrics': metrics,
            'final_reward': np.mean(metrics['reward'][-eval_episodes:]),
            'final_R_omega': np.mean(metrics['R_omega'][-eval_episodes:]),
            'final_R_psi': np.mean(metrics['R_psi'][-eval_episodes:]),
            'final_phi': np.mean(metrics['phi'][-eval_episodes:]),
        }
        
        self.results.append(result)
        
        print(f"\n✓ Complete! Final reward: {result['final_reward']:.2f}")
        print(f"  R_ω: {result['final_R_omega']:.4f}")
        print(f"  R_ψ: {result['final_R_psi']:.4f}")
        print(f"  φ: {result['final_phi']:.4f}")
        
        return result
    
    def save_results(self, filename="results.json"):
        """Save all results to JSON."""
        output_path = self.output_dir / filename
        with open(output_path, 'w') as f:
            json.dump(self.results, f, indent=2)
        print(f"\n✓ Results saved to: {output_path}")
    
    def compare_results(self):
        """Generate comparison table."""
        df = pd.DataFrame([
            {
                'Experiment': r['name'],
                'Final Reward': f"{r['final_reward']:.2f}",
                'R_ω': f"{r['final_R_omega']:.4f}",
                'R_ψ': f"{r['final_R_psi']:.4f}",
                'φ': f"{r['final_phi']:.4f}",
            }
            for r in self.results
        ])
        return df

print("✓ ExperimentRunner defined")

## Experiment 1: Baseline Comparison

Compare agents with different internal dimension sizes (0, 6, 12, 24)

In [ ]:
runner = ExperimentRunner()
env = GridWorld(size=8)

# Run experiments with different internal dimension sizes
for internal_dim in [0, 6, 12, 24]:
    config = {
        'hidden_size': 128,
        'internal_dim': internal_dim,
        'learning_rate': 3e-4,
    }
    
    runner.run_experiment(
        name=f"Baseline (dim={internal_dim})",
        env=env,
        agent_config=config,
        num_episodes=50,  # Reduced for notebook
        max_steps=100,
    )

In [ ]:
# Display comparison table
display(Markdown("## Baseline Comparison Results\n"))
display(runner.compare_results())

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for result in runner.results:
    metrics = result['metrics']
    label = result['name']
    
    axes[0, 0].plot(metrics['episode'], metrics['reward'], label=label, alpha=0.7)
    axes[0, 1].plot(metrics['episode'], metrics['R_omega'], label=label, alpha=0.7)
    axes[1, 0].plot(metrics['episode'], metrics['R_psi'], label=label, alpha=0.7)
    axes[1, 1].plot(metrics['episode'], metrics['phi'], label=label, alpha=0.7)

axes[0, 0].set_title('Reward')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_title('Internal Dimension Richness (R_ω)')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('R_ω')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].set_title('Phenomenal Binding (R_ψ)')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('R_ψ')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].set_title('Integrated Information (φ)')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('φ')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(runner.output_dir / 'baseline_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Plot saved")

## Experiment 2: Curiosity-Driven Exploration

Test different intrinsic reward weights in sparse reward environments

In [ ]:
runner2 = ExperimentRunner(output_dir="../data/curiosity_experiments")

# Create sparse reward environment
base_env = TwoRoomGridWorld(size=8, num_rooms=2)
sparse_env = SparseRewardWrapper(base_env, sparsity=0.9)

# Test different intrinsic reward weights
for intrinsic_weight in [0.0, 0.2, 0.4]:
    config = {
        'hidden_size': 128,
        'internal_dim': 12,
        'learning_rate': 3e-4,
        'intrinsic_reward_weight': intrinsic_weight,
    }
    
    runner2.run_experiment(
        name=f"Curiosity (weight={intrinsic_weight})",
        env=sparse_env,
        agent_config=config,
        num_episodes=50,
        max_steps=150,
    )

In [ ]:
# Display results
display(Markdown("## Curiosity Experiment Results\n"))
display(runner2.compare_results())

## Statistical Analysis

Perform hypothesis testing on the results:

In [ ]:
def statistical_comparison(runner, metric='reward'):
    """Compare experiments statistically."""
    print(f"\nStatistical Comparison - {metric.upper()}")
    print("=" * 60)
    
    for i in range(len(runner.results)):
        for j in range(i + 1, len(runner.results)):
            exp1 = runner.results[i]
            exp2 = runner.results[j]
            
            data1 = exp1['metrics'][metric][-10:]  # Last 10 episodes
            data2 = exp2['metrics'][metric][-10:]
            
            # T-test
            t_stat, p_value = stats.ttest_ind(data1, data2)
            
            print(f"\n{exp1['name']} vs {exp2['name']}:")
            print(f"  Mean: {np.mean(data1):.4f} vs {np.mean(data2):.4f}")
            print(f"  t-statistic: {t_stat:.4f}")
            print(f"  p-value: {p_value:.4f}")
            
            if p_value < 0.05:
                print(f"  ✓ Statistically significant difference (p < 0.05)")
            else:
                print(f"  ✗ No significant difference (p >= 0.05)")

# Run statistical tests
statistical_comparison(runner, 'reward')
statistical_comparison(runner, 'R_omega')

## Experiment 3: Multi-Agent Social Behavior

Test cooperation in Prisoner's Dilemma:

In [ ]:
# Multi-agent experiment (simplified for demo)
print("\n" + "="*60)
print("Multi-Agent Social Behavior Experiment")
print("="*60)

social_env = IteratedPrisonersDilemma(
    num_agents=2,
    internal_dim=12,
    max_steps=50,
    history_window=5,
)

# Create agents
agents = {}
for agent_name in social_env.agents:
    agents[agent_name] = PPOAgent(
        observation_space=social_env.observation_space,
        action_space=social_env.action_space,
        hidden_size=64,
        internal_dim=12,
    )

# Run episodes
cooperation_rates = {agent: [] for agent in social_env.agents}
num_episodes = 20

for episode in tqdm(range(num_episodes), desc="Social Learning"):
    obs, info = social_env.reset()
    
    for step in range(50):
        # Get actions from both agents
        actions = {}
        internal_states = {}
        
        for agent_name, agent in agents.items():
            with torch.no_grad():
                obs_tensor = torch.FloatTensor(obs[agent_name]).unsqueeze(0)
                action_logits, value, x12, m12 = agent.policy(obs_tensor)
                action_dist = torch.distributions.Categorical(logits=action_logits)
                action = action_dist.sample()
                actions[agent_name] = action.item()
                internal_states[agent_name] = {
                    'x12': x12.squeeze().numpy(),
                    'm12': m12.squeeze().numpy(),
                }
        
        # Update environment
        social_env.update_internal_states(internal_states)
        obs, rewards, terminateds, truncateds, info = social_env.step(actions)
        
        if all(terminateds.values()):
            break
    
    # Record cooperation rates
    for agent in social_env.agents:
        cooperation_rates[agent].append(info['cooperation_rate'][agent])

# Plot results
plt.figure(figsize=(10, 5))
for agent in social_env.agents:
    plt.plot(cooperation_rates[agent], label=agent, linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Cooperation Rate')
plt.title('Multi-Agent Cooperation Learning')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n✓ Multi-agent experiment complete!")
for agent in social_env.agents:
    print(f"  {agent} final cooperation rate: {cooperation_rates[agent][-1]:.2%}")

## Save All Results

In [ ]:
runner.save_results('baseline_results.json')
runner2.save_results('curiosity_results.json')

print("\n✓ All experiments complete and results saved!")

## Summary

This notebook demonstrated:

1. ✅ Automated experiment execution
2. ✅ Baseline comparisons across dimension sizes
3. ✅ Curiosity-driven exploration analysis
4. ✅ Multi-agent social behavior testing
5. ✅ Statistical hypothesis testing
6. ✅ Publication-ready visualizations

## Next Steps

- **Extend experiments**: Modify parameters and re-run
- **Add new metrics**: Track custom consciousness measures
- **Deep analysis**: Use `03_consciousness_analysis.ipynb` for detailed investigation
- **Scale up**: Increase episode counts for production experiments

Happy experimenting! 🔬